# 05 · Final model (recomputed into a new folder)

The model chosen in notebook 04 is tuned on **all 109 development boreholes** with the same procedure as the inner CV (groups = the 5 outer folds), then trained on all of them, exactly as V3.

It is saved in **this** project's folder. V3's frozen model is **never touched**; it is only read, to check that the new model predicts **exactly** the same probabilities.

**Run time:** 5-30 minutes.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
dev = cells[cells.set == "development"]
sel = json.loads((H.out_dir(S, "04_results") / "selected_model.json").read_text())
NAME, BLOCK = sel["selected"], H.block_sizes(S)[0]
OUT, PRIV = H.out_dir(S, "05_final_model"), H.out_dir(S, "05_final_model", private=True)
print("Final model:", NAME, f"| {BLOCK} m blocks | {dev.BH_Code.nunique()} boreholes, {len(dev)} cells")

In [ ]:
from tqdm import tqdm
bar = tqdm(total=H.n_trials(NAME, S), desc=f"final {NAME}", bar_format="{desc} {percentage:3.0f}%|{bar:25}| {n_fmt}/{total_fmt} trials [{elapsed}<{remaining}]")
final = H.fit_final(NAME, dev, BLOCK, S, on_trial=lambda: bar.update(1))
bar.close()
path = PRIV / f"final_model_A_{NAME}_{BLOCK}m.joblib"
joblib.dump(final["model"], path, compress=3)
rec = {"model": NAME, "block_m": BLOCK, "params": final["params"], "threshold": final["threshold"],
       "cv_macro_f1_all_development": final["inner_macro_f1"], "n_train_boreholes": int(dev.BH_Code.nunique()),
       "n_train_cells": len(dev), "top_elevation_range_m": [float(dev.Top_Elevation_m.min()), float(dev.Top_Elevation_m.max())],
       "model_file": path.name, "model_sha256": H.sha256(path)}
(OUT / "final_model.json").write_text(json.dumps(rec, indent=1, default=str))
final["trials"].to_csv(OUT / "final_model_optuna_trials.csv", index=False)
print(json.dumps(rec, indent=1, default=str))

## Same as V3's frozen model?
Both models predict every cell of the 120 Downtown boreholes; the probabilities must be identical.

In [ ]:
v3_rec = json.loads((H.v3_outputs(S, "05_final") / "final_model.json").read_text())
v3_path = H.v3_outputs(S, "05_final", "restricted") / v3_rec["model_file"]
assert H.sha256(v3_path) == v3_rec["model_sha256"], "V3's frozen model file has changed!"
v3_model = H.load_v3_model(S, v3_path)
X = cells[H.FEATURES].values
same_p = np.array_equal(final["model"].predict_proba(X)[:, 1], v3_model.predict_proba(X)[:, 1])
print("Model:", NAME == v3_rec["model"], "| parameters:", final["params"] == v3_rec["params"],
      "| threshold:", final["threshold"] == v3_rec["threshold"], f"({final['threshold']})")
print("Predictions on all", len(X), "cells IDENTICAL to V3's frozen model:", same_p)
print(f"Finished in {(time.time() - T0) / 60:.1f} min. Next: notebook 06.")